## This uses vLLM , which would be quicker

#### Cell 1 — Install vllm

In [ ]:
!pip install -q vllm


#### Cell 2 — Imports

In [2]:
import json, os, random, time
from pathlib import Path
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm.auto import tqdm

#### Cell 3 — Config

In [ ]:
INPUT_DIR = "/kaggle/input/historyfile/"
OUTPUT_DIR = "/kaggle/working/raft_outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_DISTRACTORS = 4
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507" #"Qwen/Qwen3-4B-Instruct-2507"
MAX_NEW_TOKENS = 100
MAX_MODEL_LEN = 2048

KAGGLE_DATASET = "kanav608/raft-intermediate"
COMMIT_EVERY_N_FILES = 1

#### Cell 4 — Kaggle auth (via Secrets, not hardcoded key)

In [ ]:
from kaggle_secrets import UserSecretsClient
from kaggle.api.kaggle_api_extended import KaggleApi

user_secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = 'kanav608'
os.environ['KAGGLE_KEY'] = 'alpha_beta'

api = KaggleApi()
api.authenticate()

#### Cell 5 — Load model with vLLM

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

llm = LLM(
    model=MODEL_ID,
    dtype="float16",
    gpu_memory_utilization=0.85,
    max_model_len=MAX_MODEL_LEN,
    trust_remote_code=True,
)

print("✅ vLLM engine loaded.")

#### Cell 6 — Batched generation via vLLM

In [6]:
def batch_generate_qa_vllm(prompts, max_tokens=MAX_NEW_TOKENS):
    formatted = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": p}],
            tokenize=False,
            add_generation_prompt=True,
        )
        for p in prompts
    ]
    params = SamplingParams(temperature=0.0, max_tokens=max_tokens)
    outputs = llm.generate(formatted, params, use_tqdm=True)
    return [o.outputs[0].text.strip() for o in outputs]

#### Cell 7 — Upload helper (unchanged)

In [ ]:
def upload_to_dataset(dir_path, message="Update RAFT files"):
    metadata_path = os.path.join(dir_path, "dataset-metadata.json")
    if not os.path.exists(metadata_path):
        metadata = {
            "id": KAGGLE_DATASET,
            "title": "RAFT training examples",
            "licenses": [{"name": "CC0-1.0"}]
        }
        with open(metadata_path, "w", encoding="utf-8") as f:
            json.dump(metadata, f)
        print("📄 Created dataset-metadata.json")

    try:
        api.dataset_create_version(folder=dir_path, version_notes=message, quiet=False)
        print(f"Dataset version committed: {message}")
    except Exception as e:
        print(f"Failed to upload: {e}")

#### Cell 8 — process_file (same logic, new generation call)

In [8]:
def process_file(file_path, output_dir):
    with open(file_path, "r", encoding="utf-8") as f:
        chunks = [json.loads(line) for line in f if line.strip()]

    if len(chunks) < 5:
        return

    contents = [c["content"] for c in chunks]
    prompts = []
    meta = []

    for idx, cdata in enumerate(chunks):
        text = cdata["content"]
        if len(text) < 100:
            continue
        perspective = cdata.get("historian_perspective", "")
        historian = cdata.get("historian", "")
        domain = cdata.get("expert_domain", "")

        hint = ""
        if perspective and historian:
            hint = f"You are a {perspective} historian ({historian}). "
        elif perspective:
            hint = f"You are a {perspective} historian, who focus on {domain} period. "

        prompt = f"""{hint}Read the following text and generate a question whose answer is directly contained in the text. The question should reflect the historical perspective if mentioned. Then provide the exact answer.

Text:
\"\"\"
{text}
\"\"\"

Reply with a JSON object containing exactly two keys: "question" and "answer".
Only return the JSON, no extra text."""
        prompts.append(prompt)
        meta.append((idx, cdata))

    if not prompts:
        return

    raw = batch_generate_qa_vllm(prompts)

    examples = []
    for i, r in enumerate(raw):
        r_clean = r.replace("```json", "").replace("```", "").strip()
        try:
            qa = json.loads(r_clean)
            q, a = qa.get("question", ""), qa.get("answer", "")
        except Exception:
            continue
        if not q or not a:
            continue

        idx, orig = meta[i]
        relevant = contents[idx]
        other_idx = [j for j in range(len(contents)) if j != idx]
        random.shuffle(other_idx)
        distractors = []
        al = a.strip().lower()
        for j in other_idx:
            if al not in contents[j].lower():
                distractors.append(contents[j])
            if len(distractors) == NUM_DISTRACTORS:
                break
        if len(distractors) < NUM_DISTRACTORS:
            continue

        docs = [relevant] + distractors
        random.shuffle(docs)
        doc_list = [f"### Document [{k+1}]: {d}" for k, d in enumerate(docs)]
        ex = {
            "instruction": "You are a helpful assistant. Use the provided documents to answer the question. If the answer cannot be found in the documents, say 'I don't know'.",
            "documents": doc_list,
            "question": q,
            "output": a,
            "perspective": orig.get("historian_perspective", ""),
            "historian": orig.get("historian", ""),
            "expert_domain": orig.get("expert_domain", ""),
        }
        examples.append(ex)

    out_name = file_path.stem + "_raft.jsonl"
    out_path = os.path.join(output_dir, out_name)
    with open(out_path, "w", encoding="utf-8") as fout:
        for e in examples:
            fout.write(json.dumps(e, ensure_ascii=False) + "\n")

    print(f"   → Wrote {len(examples)} examples to {out_name}")

#### Cell 9 — Main loop

In [ ]:
all_files = sorted(Path(INPUT_DIR).rglob("*.jsonl"))
print(f"Found {len(all_files)} .jsonl files.\n")

existing_outputs = set(os.listdir(OUTPUT_DIR)) if os.path.exists(OUTPUT_DIR) else set()


target_files = {
     "Fall_Of_The_Mughal_Empire_Vol_",
    "Al-Hind_Vol_",
    "Rajasthan_Through_the_Ages_Vol_",

}

selected_files = [f for f in all_files if f.stem.startswith(tuple(target_files))]
print(f"Selected {len(selected_files)} target files.\n")

processed_count = 0
t0 = time.time()

for fpath in selected_files:
    out_name = fpath.stem + "_raft.jsonl"
    if out_name in existing_outputs:
        print(f"⏩ Skipping {fpath.name} (already processed)")
        continue

    processed_count += 1
    print(f"\n--- Processing file {processed_count}/{len(selected_files)}: {fpath.name} ---")
    process_file(fpath, OUTPUT_DIR)

    if processed_count % COMMIT_EVERY_N_FILES == 0:
        print("📦 Committing current outputs to Kaggle dataset...")
        upload_to_dataset(OUTPUT_DIR, f"Processed {processed_count} target files")

if len(os.listdir(OUTPUT_DIR)) > 0:
    print("\n📦 Final commit to dataset...")
    upload_to_dataset(OUTPUT_DIR, f"All done – {len(os.listdir(OUTPUT_DIR))} files")

print(f"\n🏁 Done in {time.time()-t0:.1f}s. Outputs are in {OUTPUT_DIR}")